# 第十七课｜外部内存是什么？

片上 RAM 靠近逻辑，但容量有限。真实 connectome 的 synapse store 很快会超过片上容量，于是需要：

> **把大量数据放到 FPGA 芯片外的内存。**

主要新概念：**外部 DDR 有独立访问延迟，连续 burst 与零散 random access 的成本不同。**

## 1. 概念账本

**已经知道：** memory hierarchy、latency、bandwidth、synapse records。

**今天学习：**
- **双倍数据速率同步动态随机存储器（Double Data Rate Synchronous Dynamic Random-Access Memory, DDR SDRAM）**；
- **burst**：一次事务连续传输多个相邻数据 beat；
- sequential access 与 random access。

**只预告：** AXI 是访问这类内存的常见上层接口之一。

## 2. 为什么不把所有 synapse 都放片上？

```mermaid
flowchart LR
  ENG["synapse engine"] --> CTRL["platform memory controller / IP"]
  CTRL --> DDR["external DDR memory"]
```

外部 DDR 提供更大容量，但访问路径更长、控制更复杂。项目使用平台提供的 controller/IP，不要求学生手写 DDR PHY。

## 3. 为什么连续访问通常更友好？

很多访问存在固定启动成本。相邻地址若能合并成 burst，这个成本可以由多份数据共同分担；random access 更容易频繁重新开始事务。

## 4. 一个教学 burst model

把地址看成 word index。只有当 `current == previous + 1` 且当前 burst 未满时，才继续同一个 burst。这个模型只观察 access pattern，不是 DDR controller。

## 5. Run：连续地址与 random-like 地址

先预测 8 个连续 word、最大 burst 长度 4 时需要几个 burst。

In [ ]:
def estimate_bursts(addresses, max_burst_words):
    if not addresses:
        return 0
    bursts = 1
    run_length = 1
    for previous, current in zip(addresses, addresses[1:]):
        if current == previous + 1 and run_length < max_burst_words:
            run_length += 1
        else:
            bursts += 1
            run_length = 1
    return bursts

sequential = list(range(8))
random_like = [0, 9, 2, 14, 7, 20, 1, 30]

print("sequential bursts:", estimate_bursts(sequential, 4))
print("random-like bursts:", estimate_bursts(random_like, 4))


## 6. Observe

8 个连续地址组成两个 4-word burst；random-like 地址几乎每个都需要新 burst。总 bytes 相同，访问模式仍会改变有效成本。

## 7. integrity test 为什么先于性能？

DDR hello-world 先写入一批已知值，再可靠读回。只有 read/write integrity 稳定后，benchmark bandwidth 才有意义。

## 8. Try It

把 sequential 改成 `list(range(10))`，max burst 仍为 4；再把 max burst 改成 8。分别先预测 burst 数。

## 9. 作业

[第 17 课作业：估算 sequential/random access 的 burst 成本](../../exercises/zh/17_external_memory_ddr.ipynb)

## 10. AI Task

让 AI 解释“同样 1 KB 数据，连续读取与随机读取为什么可能不同”。检查它不要误说 random access 不能工作；这里讨论的是效率。

## 11. Human Check

解释 DDR 为什么解决容量但引入访问成本；burst 为什么适合连续数据；random/sequential 为什么影响有效 bandwidth；为什么不手写 DDR PHY。

## 12. Engineering Handoff

对应 `RMD-014`：通过平台 controller/IP 做稳定 read/write + integrity test。

## 13. 项目追踪 Project Trace

- Lesson: `LSN-017`
- Mapping: `RMD-014`
- First proof: DDR read/write integrity
- Performance preview: sequential vs random vs burst

## 14. Exit Ticket

你能解释外部 DDR 为什么容量大但访问成本不同，并能根据简单地址序列判断 burst 数。